In [ ]:
# 导入必要的库
import operator
from typing import Annotated, Literal, List, Dict, Any
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint import InMemorySaver
from langgraph.types import interrupt, Command

# ================= 状态定义层 =================
# 定义 State（状态），是整个图执行过程中的"共享全局内存"。
class AgentState(TypedDict):
    """
    TypedDict 类型用于定义状态的结构，确保类型安全。
    字段使用 Annotated 并配合 Reducer 来定义更新策略。
    """
    # -- 控制流字段 --
    next_action: Literal["assistant_reply", "human_review", "end"]  # 决定下一步去哪个节点
    
    # -- 数据载体字段 --
    question: str                # 用户输入的问题
    assistant_answer: str       # AI 助手的回复
    final_answer: str           # 审核后的最终回复
    
    # -- 系统字段 --
    language: str                # 问题语言
    need_human_review: bool      # 是否需要人工复核
    retry_count: int             # 重试次数，用于循环控制
    
    # -- 历史累积字段（使用 Reducer 实现自动拼接）--
    # Annotated: 为类型添加元数据。此处通过 operator.add 作为 Reducer，
    # 告诉 LangGraph：当多个节点都向此字段追加内容时，自动拼接成列表，而不是覆盖。
    messages: Annotated[List[str], operator.add]

# ================= 节点定义层 =================
# 节点是执行具体任务的单元。每个节点函数接收完整的 state，并返回一个部分 (Partial) 的 state 更新。
def analyze_node(state: AgentState) -> dict:
    """
    节点 1: 分析意图
    - 目的：模拟对用户问题的分析过程。
    - 逻辑：检查问题中的关键词，以确定语言和是否需要人工复核。
    """
    print(f"[分析意图] 正在分析问题: {state['question']}")
    # 简单的关键词匹配逻辑
    if "投诉" in state["question"]:
        need_review = True
    else:
        need_review = False
    
    language = "en" if "hello" in state["question"].lower() else "zh"
    
    # 返回状态更新：只返回需要修改的字段
    return {
        "language": language,
        "need_human_review": need_review,
        "messages": [f"分析节点完成: 语言是 {language}, 需要复核: {need_review}"],
        "retry_count": state["retry_count"] + 1  # 重试次数+1
    }

def assistant_node(state: AgentState) -> dict:
    """
    节点 2: AI 助手处理
    - 目的：模拟 AI 助手生成回复的过程。
    - 核心：为不同类型的用户问题生成一个预设的 AI 回复。
    """
    print("[AI助手] 正在生成回复...")
    # 根据问题内容生成回复
    if "退货" in state["question"]:
        answer = "亲，根据我们的7天无理由退货政策，您可以申请退货。"
    elif "投诉" in state["question"]:
        answer = "非常抱歉给您带来了不好的体验，我们会立刻处理您的问题。"
    else:
        answer = "您好，请问有什么可以帮您？"
    
    return {
        "assistant_answer": answer,
        "messages": [f"AI助手节点生成回复: {answer}"]
    }

def human_review_node(state: AgentState) -> Command[Literal["assistant_reply", "end"]]:
    """
    节点 3: 人工复核节点 (Human-in-the-Loop)
    - 目的：通过 Command 和 interrupt() 暂停图执行，等待外部输入（如人工客服的审核结果）。
    - 重点：使用 `interrupt()` 挂起图并接收人工审核结果，使用 `Command` 携带人工审核后的结果并路由到下一个节点。
    """
    print("[人工复核] 暂停执行，等待人工审核...")
    
    # 在等待人工审核期间，使用 `interrupt` 函数挂起图的执行
    # 将当前 AI 生成的回复和问题一起传递给人工审核的接口
    human_feedback = interrupt({
        "question": state["question"], 
        "ai_answer": state["assistant_answer"]
    })
    
    # 假设外部系统在恢复执行时传入了审核后的最终回复
    # `interrupt` 的返回值即是外部系统传入的值
    final_answer = human_feedback.get("final_answer", state["assistant_answer"])
    print(f"[人工复核] 收到审核结果: {final_answer}")
    
    # `Command` 对象允许在返回状态的同时，进行高级控制流操作
    # 这里我们使用 `goto` 参数，指示图在执行完该节点后直接路由到 `END` 节点
    return Command(
        update={"final_answer": final_answer, "messages": ["人工复核完成"]},
        goto=END
    )

# ================= 条件路由定义 =================
def router(state: AgentState) -> Literal["human_review", "assistant_reply", "end"]:
    """
    条件函数
    - 作用：决定在 analyze_node 和 assistant_node 之后，下一步应该去哪。
    - 逻辑：基于状态中的字段，返回下一个节点的名称。
    """
    print(f"[路由决策] next_action 当前值: {state.get('next_action')}")
    if state.get("next_action") == "end":
        return "end"
    
    # 关键逻辑：如果之前在 analyze_node 中检测到需要人工复核，则路由到 human_review_node
    if state.get("need_human_review"):
        return "human_review"
    else:
        return "assistant_reply"

# ================= 图构建与编译 =================
# 1. 初始化图对象
# StateGraph 是 LangGraph 的核心构造器，它管理着整个图的结构。
# 我们需要将之前定义的 AgentState 传递给它，以便它知道整个图的状态结构。
graph_builder = StateGraph(AgentState)

# 2. 添加节点
# `add_node` 方法用于将功能节点注册到图中。
# 第一个参数是节点的唯一标识符（字符串），第二个参数是节点函数。
graph_builder.add_node("analyze", analyze_node)
graph_builder.add_node("assistant_reply", assistant_node)
graph_builder.add_node("human_review", human_review_node)

# 3. 添加边
# `add_edge` 用于添加静态边，表示从一个节点无条件地直接流向另一个节点。
# 这里代表一旦 analyze_node 执行完毕，就会无条件执行 router 条件函数。
# 但实际上，我们需要在 analyze_node 后加一个条件边来实现路由。

# `add_conditional_edges` 用于添加动态的、基于逻辑的边。
# 第一个参数是起始节点 ("analyze")，第二个参数是路由函数 (router)。
graph_builder.add_conditional_edges("analyze", router)

# 静态边：如果 AI 助手处理完成，则结束流程。
graph_builder.add_edge("assistant_reply", END)

# 设置图的入口点。START 是一个特殊节点，代表图的开始。
graph_builder.set_entry_point("analyze")

# 4. 编译图
# `compile` 方法将我们定义的蓝图图结构，编译成一个可执行的、线程安全的运行时对象。
# `checkpointer` 参数是 LangGraph 实现记忆和"时间旅行"的关键。
# `InMemorySaver` 是一个将状态保存在内存中的检查点保存器，它使得图可以从中断点恢复执行。
memory = InMemorySaver()
graph = graph_builder.compile(checkpointer=memory)

# ================= 执行演示 =================
# 为了在多次调用中维持会话状态，LangGraph 引入了 `thread_id` 的概念。
config = {"configurable": {"thread_id": "user_12345"}}

# 场景一：AI 自动处理流程（不涉及人工审核）
print("\n=== 场景一：AI 自动处理 ===")
user_input = {"question": "我想咨询一下退货政策。", "retry_count": 0}
for event in graph.stream(user_input, config):
    print(f"流式事件: {event}")

# 场景二：人机协同流程（需要人工审核）
# 为演示 Human-in-the-Loop，需要新的会话 ID 或重置状态，这里创建一个新的线程。
new_config = {"configurable": {"thread_id": "user_complaint_123"}}
print("\n=== 场景二：人机协同 ===")
user_input_complaint = {"question": "我要投诉你们的服务质量！", "retry_count": 0}

try:
    # 第一次执行，会触发 interrupt
    for event in graph.stream(user_input_complaint, new_config):
        print(f"流式事件: {event}")
except Exception as e:
    # 注意：生产环境中不应依赖捕获通用 Exception 来处理 interrupt
    print("图形已中断，等待人工输入...")

# 获取当前中断状态
current_state = graph.get_state(new_config)
print(f"当前状态: {current_state.values}")

# 人工输入结果，并使用 `Command` 恢复执行
# `Command(resume=...)` 恢复 `interrupt()` 所在节点，提供数据并继续执行
print("人工客服介入...")
human_response = {"final_answer": "您的投诉我们已收到，专员会致电给您。"}
for event in graph.stream(Command(resume=human_response), new_config):
    print(f"恢复后的事件: {event}")

final_state = graph.get_state(new_config)
print(f"\n最终状态：\n用户问题：{final_state.values['question']}\n"
      f"AI初始回复：{final_state.values['assistant_answer']}\n"
      f"人工最终回复：{final_state.values['final_answer']}")